In [1]:
import pandas as pd
import os
import glob
import subprocess

In [2]:
def reformat_dosage(fn='/data100t1/home/wanying/BioVU/20260305_AGD_250k_HLA_typing/output/kourami_result_combined_agd250k/imgt_v3.64/AGD250k.kourami.imgt_v3.64.HLA_A.txt',
                    output_fn='output.txt'):
    '''
    Reformat the HLA output so that each row represent a sample, and each column represent a HLA allele
    This convertion is necessary for downstream asscosiaiton tests
    
    '''
    df = pd.read_csv(fn, sep='\t').dropna(subset=['allele1', 'allele2'])
    n_valid_samples = len(df) # Track number of valid samples
    columns = list(set(list(df['allele1']) + list(df['allele2'])))
    print('# N unique HLA alleles:', len(columns))
    df_reformat = pd.DataFrame(columns=['grid']+columns)
    df_reformat['grid'] = df['grid']
    for i, allele in enumerate(columns):
        mask_a1 = df['allele1']==allele
        mask_a2 = df['allele2']==allele
        df_reformat[allele] = 0
        df_reformat.loc[mask_a1, allele] += 1
        df_reformat.loc[mask_a2, allele] += 1
        print(f'\r# Processed alleles: {i+1}', end='', flush=True)
    print(f'\r# Processed alleles: {i+1}')
    print('# Save file', end='')
    df_reformat.to_csv(output_fn, sep='\t', index=False)
    print('# gzip compress')
    subprocess.run(f'gzip -f {output_fn}', shell=True, check=True)
    return n_valid_samples


In [3]:
result_folder = '/data100t1/home/wanying/BioVU/20260305_AGD_250k_HLA_typing/output/kourami_result_combined_agd250k'
lst_n_valid_samples, lst_ref_versions, lst_hla_genes = [], [], []
for ref_version in ['3.24', '3.59', '3.64']:
    print('\n#', '#'*20, f'IMGT database v{ref_version}', '#'*20)
    for fn in glob.glob(f'{result_folder}/imgt_v{ref_version}/AGD250k.kourami.imgt_v*.HLA_*.txt'):
        hla_gene = fn.split('/')[-1].split('.')[4]
        print('\n#', '-'*10, hla_gene, '-'*10)
        output_fn = f'{result_folder}/imgt_v{ref_version}/AGD250k.kourami.imgt_v3.24.{hla_gene}.dosage.tsv'
        n_valid_samples = reformat_dosage(fn=fn,
                                          output_fn=output_fn)
        lst_n_valid_samples.append(n_valid_samples)
        lst_ref_versions.append(ref_version)
        lst_hla_genes.append(hla_gene)




# #################### IMGT database v3.24 ####################

# ---------- HLA_B ----------
# N unique HLA alleles: 2994
# Processed alleles: 2994
# Save file# gzip compress

# ---------- HLA_DQA1 ----------
# N unique HLA alleles: 27
# Processed alleles: 27
# Save file# gzip compress

# ---------- HLA_DRB1 ----------
# N unique HLA alleles: 875
# Processed alleles: 875
# Save file# gzip compress

# ---------- HLA_C ----------
# N unique HLA alleles: 2448
# Processed alleles: 2448
# Save file# gzip compress

# ---------- HLA_F ----------
# N unique HLA alleles: 4
# Processed alleles: 4
# Save file# gzip compress

# ---------- HLA_A ----------
# N unique HLA alleles: 1524
# Processed alleles: 1524
# Save file# gzip compress

# ---------- HLA_L ----------
# N unique HLA alleles: 4
# Processed alleles: 4
# Save file# gzip compress

# ---------- HLA_DPA1 ----------
# N unique HLA alleles: 65
# Processed alleles: 65
# Save file# gzip compress

# ---------- HLA_DQB1 ----------
# N unique

In [5]:
df_result_summary = pd.DataFrame({'imgt_version':lst_ref_versions, 'hla_gene':lst_hla_genes, 'number_of_valid_samples':lst_n_valid_samples})
display(df_result_summary)

output_fn = '/data100t1/home/wanying/BioVU/20260305_AGD_250k_HLA_typing/output/kourami_result_combined_agd250k/summary.txt'
df_result_summary.to_csv(output_fn, sep='\t', index=False)

,imgt_version,hla_gene,number_of_valid_samples
0,3.24,HLA_B,250095
1,3.24,HLA_DQA1,250166
2,3.24,HLA_DRB1,250166
3,3.24,HLA_C,250033
4,3.24,HLA_F,250084
5,3.24,HLA_A,250152
6,3.24,HLA_L,250013
7,3.24,HLA_DPA1,250094
8,3.24,HLA_DQB1,250166
9,3.24,HLA_J,250094


9